# 12 — ESM-2 3B Flow Model Experiment

**Question:** Does the direction reversal get stronger with ESM-2 3B (2560d)?

- 35M (480d): ρ=+0.229 (positive)
- 650M (1280d): ρ=-0.219 (negative)
- **3B (2560d): ???**

**Requirements:** A100 GPU (40GB+ VRAM)

**Pipeline:**
1. Generate training data with ESM-2 3B embeddings
2. Train flow model (~2h on A100)
3. Score Aβ42 DMS mutations
4. Compare direction and magnitude with 35M/650M
5. Embedding probe analysis for 3B

In [ ]:
# CELL 0: Setup
import os, sys
os.chdir('/content')
if not os.path.exists('brain_idp_flow'):
    !git clone https://github.com/layeredlabs06/brain-idp-flow.git brain_idp_flow
else:
    !cd brain_idp_flow && git pull origin master
os.chdir('brain_idp_flow')
!pip install -e . -q
!pip install fair-esm scipy scikit-learn -q
sys.path.insert(0, '/content/brain_idp_flow/src')

# train.npz 가 없으면 업로드 받기
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.npz'):
    print('⚠ data/train.npz 없음 — 파일 업로드 필요')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.npz'):
            import shutil
            shutil.move(fname, 'data/train.npz')
            print(f'Saved to data/train.npz')
            break

import torch, numpy as np, yaml, pickle, os, time
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name()
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}, VRAM: {vram:.1f} GB')

In [ ]:
# CELL 1: Drive mount + check for existing 3B checkpoint
from google.colab import drive
drive.mount('/content/drive')

CKPT_3B = '/content/drive/MyDrive/brain_idp_flow_ckpts_3b/ckpt_best.pt'
TRAIN_NPZ = 'data/train.npz'
SCORE_CACHE = '/content/drive/MyDrive/brain_idp_flow_scored/all_scores.pkl'

HAS_3B_CKPT = os.path.exists(CKPT_3B)
print(f'3B checkpoint exists: {HAS_3B_CKPT}')
print(f'Training data exists: {os.path.exists(TRAIN_NPZ)}')

In [ ]:
# CELL 2: Generate training data with ESM-2 3B embeddings (skip if train.npz exists)
from brain_idp_flow.features.seq_embed import ESM2Embedder
from brain_idp_flow.targets import load_targets
from brain_idp_flow.data.dataset import ProteinEnsembleDataset

embedder_3b = ESM2Embedder(model_name='esm2_t36_3B_UR50D', device=device)
print(f'ESM-2 3B loaded, embed_dim={embedder_3b.embed_dim}')

# Quick test
test_emb = embedder_3b.embed_single('DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA')
print(f'Test embedding shape: {test_emb.shape}')  # should be (42, 2560)

In [ ]:
# CELL 3: Train 3B flow model (skip if checkpoint exists)
if not HAS_3B_CKPT:
    from brain_idp_flow.train import train
    from brain_idp_flow.data.dataset import ProteinEnsembleDataset

    with open('configs/flow_3b.yaml') as f:
        config_3b = yaml.safe_load(f)

    # Load dataset
    ds = ProteinEnsembleDataset(config_3b['data']['train_npz'])
    unique_ids = sorted(set(int(sid) for sid in ds.seq_ids))
    print(f'Dataset: {len(unique_ids)} unique sequences, {len(ds)} samples')

    # Pre-compute ESM-2 3B embeddings for all training sequences
    print('Pre-computing ESM-2 3B embeddings...')
    seq_embeddings = {}
    for seq_id in unique_ids:
        idx = next(i for i, sid in enumerate(ds.seq_ids) if int(sid) == seq_id)
        sample = ds[idx]
        L = sample['coords'].shape[0]

        if hasattr(ds, 'sequences') and seq_id < len(ds.sequences):
            emb = embedder_3b.embed_single(ds.sequences[seq_id]).cpu()
        else:
            emb = torch.zeros(L, 2560)

        seq_embeddings[seq_id] = emb
        if len(seq_embeddings) % 10 == 0:
            print(f'  {len(seq_embeddings)}/{len(unique_ids)}')

    print(f'Prepared {len(seq_embeddings)} embeddings')

    print('Starting 3B training...')
    t0 = time.time()
    best_ckpt = train(config_3b, seq_embeddings, device)
    elapsed = (time.time() - t0) / 60
    print(f'Training done in {elapsed:.1f} min')

    # Save to Drive
    os.makedirs(os.path.dirname(CKPT_3B), exist_ok=True)
    import shutil
    shutil.copy(str(best_ckpt), CKPT_3B)
    # Drive에 train.npz도 백업
    shutil.copy('data/train.npz', '/content/drive/MyDrive/brain_idp_flow_scored/train.npz')
    print(f'Saved checkpoint to {CKPT_3B}')
else:
    print(f'Using existing checkpoint: {CKPT_3B}')

In [ ]:
# CELL 4: Score Aβ42 DMS with 3B flow model
from brain_idp_flow.sample import load_model, sample_ensemble
from brain_idp_flow.geometry.metrics import radius_of_gyration
from brain_idp_flow.data.dms_loader import load_seuma_dms, ABETA42_WT
from tqdm.auto import tqdm

with open('configs/flow_3b.yaml') as f:
    config_3b = yaml.safe_load(f)

model_3b = load_model(config_3b, CKPT_3B, device)
print('3B flow model loaded')

# WT ensemble
wt_emb_3b = embedder_3b.embed_single(ABETA42_WT)
wt_ens_3b = sample_ensemble(
    model_3b, wt_emb_3b, mut_pos=0, mut_aa=0,
    n_samples=64, n_steps=50, device=device, batch_size=16,
)
wt_rg_3b = radius_of_gyration(torch.from_numpy(wt_ens_3b)).mean().item()
print(f'WT Rg (3B): {wt_rg_3b:.2f}')

# Score all mutations
os.makedirs('data/dms', exist_ok=True)
dms_data = load_seuma_dms(cache_dir='data/dms')
AA_TO_IDX = {aa: i + 1 for i, aa in enumerate('ACDEFGHIKLMNPQRSTVWY')}

scored_3b = []
for i, d in enumerate(tqdm(dms_data, desc='Scoring 3B')):
    mut_seq = list(ABETA42_WT)
    mut_seq[d['pos'] - 1] = d['mt']
    mut_seq = ''.join(mut_seq)
    mut_emb = embedder_3b.embed_single(mut_seq)
    aa_idx = AA_TO_IDX.get(d['mt'], 0)

    mut_ens = sample_ensemble(
        model_3b, mut_emb, mut_pos=d['pos'], mut_aa=aa_idx,
        n_samples=32, n_steps=50, device=device, batch_size=16,
    )
    mut_rg = radius_of_gyration(torch.from_numpy(mut_ens)).mean().item()
    scored_3b.append({**d, 'delta_rg_3b': mut_rg - wt_rg_3b})

print(f'Scored {len(scored_3b)} mutations')

In [ ]:
# CELL 5: Embedding probe — 3B layers vs nucleation
from brain_idp_flow.analysis.embedding_analysis import per_layer_rg_probe

# All mutant sequences
mut_sequences = []
for d in dms_data:
    seq = list(ABETA42_WT)
    seq[d['pos'] - 1] = d['mt']
    mut_sequences.append(''.join(seq))

ns = np.array([d['nucleation_score'] for d in dms_data])

# Per-layer probe for 3B
print('Running per-layer probes for ESM-2 3B...')
probe_3b = per_layer_rg_probe(embedder_3b, mut_sequences, ns)
print(f'Probed {len(probe_3b)} layers')
for layer_info in probe_3b:
    print(f"  Layer {layer_info['layer']:>3}: ρ={layer_info['rho']:.3f}")

In [ ]:
# CELL 6: Three-scale comparison
# Load 35M and 650M scores from cache
with open(SCORE_CACHE, 'rb') as f:
    cached = pickle.load(f)
scored_dms = cached['scored_dms']
scored_650m = cached.get('scored_650m', [])

ns_ab = np.array([d['nucleation_score'] for d in scored_dms])
drg_35m = np.array([d['delta_rg'] for d in scored_dms])
drg_650m = np.array([d['delta_rg_650m'] for d in scored_650m]) if scored_650m else None
drg_3b = np.array([d['delta_rg_3b'] for d in scored_3b])

rho_35m, p_35m = spearmanr(drg_35m, ns_ab)
rho_3b, p_3b = spearmanr(drg_3b, ns_ab)

print('='*60)
print('THREE-SCALE COMPARISON')
print('='*60)
print(f"{'Scale':<15} {'d_seq':<8} {'ρ':>8} {'p-value':>12} {'Direction':>10}")
print('-'*55)
print(f"{'35M':<15} {'480':<8} {rho_35m:>8.3f} {p_35m:>12.2e} {'Positive':>10}")
if drg_650m is not None:
    rho_650m, p_650m = spearmanr(drg_650m, ns_ab)
    print(f"{'650M':<15} {'1280':<8} {rho_650m:>8.3f} {p_650m:>12.2e} {'Negative':>10}")
direction_3b = 'Positive' if rho_3b > 0 else 'Negative'
print(f"{'3B':<15} {'2560':<8} {rho_3b:>8.3f} {p_3b:>12.2e} {direction_3b:>10}")

In [ ]:
# CELL 7: Visualization — three-scale scatter + probe comparison
os.makedirs('results/3b', exist_ok=True)
is_fad = np.array([d.get('is_fad', False) for d in scored_dms])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, drg, rho, p, title in [
    (axes[0], drg_35m, rho_35m, p_35m, 'ESM-2 35M (480d)'),
    (axes[1], drg_650m, *spearmanr(drg_650m, ns_ab), 'ESM-2 650M (1280d)') if drg_650m is not None else (axes[1], drg_3b, rho_3b, p_3b, ''),
    (axes[2], drg_3b, rho_3b, p_3b, 'ESM-2 3B (2560d)'),
]:
    if drg is None:
        ax.set_visible(False)
        continue
    ax.scatter(drg[~is_fad], ns_ab[~is_fad], s=10, alpha=0.4, c='steelblue')
    if is_fad.any():
        ax.scatter(drg[is_fad], ns_ab[is_fad], s=80, facecolors='none',
                   edgecolors='red', linewidth=2, label='fAD')
    ax.set_xlabel('ΔRg'); ax.set_ylabel('Nucleation Score')
    ax.set_title(f'{title}\nρ={rho:.3f}, p={p:.2e}')
    ax.legend(fontsize=9)

fig.suptitle('Three-Scale Comparison: ΔRg vs Nucleation Score', fontsize=14)
fig.tight_layout()
fig.savefig('results/3b/three_scale_scatter.png', dpi=300)
plt.show()

# Copy to Drive
try:
    import shutil
    shutil.copy('results/3b/three_scale_scatter.png',
                '/content/drive/MyDrive/brain_idp_flow_results/three_scale_scatter.png')
except:
    pass

In [ ]:
# CELL 8: Save 3B scores to Drive
save_path = '/content/drive/MyDrive/brain_idp_flow_scored/scores_3b.pkl'
with open(save_path, 'wb') as f:
    pickle.dump({
        'scored_3b': scored_3b,
        'probe_3b': probe_3b,
        'rho_3b': rho_3b,
        'p_3b': p_3b,
    }, f)
print(f'Saved to {save_path}')

print('\n' + '='*60)
print('SUMMARY')
print('='*60)
print(f'3B ΔRg vs nucleation: ρ={rho_3b:.3f}, p={p_3b:.2e}')
print(f'Direction: {direction_3b}')
print(f'Reversal strengthens with scale: {abs(rho_3b) > abs(rho_35m) if rho_3b < 0 else "No reversal"}')
print('\nDone!')